# 06. Supervised Field Adaptation — ResNet18

**Research question:** Does fine-tuning with a limited number of labeled PlantDoc images improve the cross-domain performance of ResNet18 classifiers previously trained on PlantVillage?

**Experiments:**

- **A1 — Baseline → Field:** Adapt the original ResNet18 baseline.
- **A2 — Geometric → Field:** Adapt the ResNet18 model trained with geometric augmentation.

Both experiments use the same 380 PlantDoc adaptation-training images, 96 PlantDoc development images, label order, preprocessing, and adaptation schedule. **The original 47-image Field Test is not used for training, checkpoint selection, or inference in this notebook.**

**Interpretation boundary:** Field Dev is a development set; improvements measured here are exploratory and are not an independent confirmatory estimate.

## 1. Runtime and dependencies

Run this notebook using a **Google Colab GPU runtime**. Data exploration, split preparation, and final report assembly can remain on the local machine.

In [9]:
%pip install -q datasets pandas scikit-learn matplotlib Pillow ImageHash

import sys
import torch
import torchvision

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Select a Colab GPU runtime before training."
print("GPU:", torch.cuda.get_device_name(0))

Python: 3.12.13
PyTorch: 2.10.0+cu128
TorchVision: 0.25.0+cu128
CUDA available: True
GPU: Tesla T4


## 2. Repository synchronization

In [10]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/thutrang147/plant-disease-diagnosis.git"
PROJECT_ROOT = Path("/content/plant-disease-diagnosis")

if not PROJECT_ROOT.is_dir():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)

GIT_SHA = subprocess.check_output(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], text=True
).strip()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Repository:", PROJECT_ROOT)
print("Git commit:", GIT_SHA)

Repository: /content/plant-disease-diagnosis
Git commit: 6ab87eda3e9ddba6dece1b12b8708f9f2ba11801


## 3. Configuration and fixed label mapping

Restore the **previously trained** checkpoints from Google Drive. The source model architectures and label order must match the five-class experiment. Keep outputs from A1 and A2 separate.

In [11]:
from google.colab import drive
import hashlib
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18
from torchvision.transforms import InterpolationMode
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

from src.dataset import PlantDiseaseDataset

drive.mount("/content/drive")

SEED = 42
BATCH_SIZE = 32
HEAD_EPOCHS = 2
LAST_BLOCK_EPOCHS = 6
EARLY_STOP_PATIENCE = 3
NUM_WORKERS = 0
DEVICE = torch.device("cuda")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
DRIVE_ROOT = Path("/content/drive/MyDrive/plant-disease-diagnosis")
CHECKPOINTS = {
    "A1_baseline": DRIVE_ROOT / "resnet18_baseline" / "resnet18_lab_best.pt",
    "A2_geometric": DRIVE_ROOT / "resnet18_aug_geometric" / "resnet18_best.pt",
}
OUTPUT_ROOT = DRIVE_ROOT / "field_adaptation"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

for path in CHECKPOINTS.values():
    if not path.is_file():
        raise FileNotFoundError(f"Checkpoint missing on Google Drive: {path}")

label_map = json.loads((SPLITS_DIR / "label_map.json").read_text(encoding="utf-8"))
CLASS_NAMES = [name for name, idx in sorted(label_map.items(), key=lambda item: item[1])]
assert len(CLASS_NAMES) == 5 and sorted(label_map.values()) == list(range(5))
print("Class order:", CLASS_NAMES)
print("Checkpoint files:", *[str(p) for p in CHECKPOINTS.values()], sep="\n - ")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Class order: ['Healthy', 'Early Blight', 'Late Blight', 'Bacterial Spot', 'Septoria Leaf Spot']
Checkpoint files:
 - /content/drive/MyDrive/plant-disease-diagnosis/resnet18_baseline/resnet18_lab_best.pt
 - /content/drive/MyDrive/plant-disease-diagnosis/resnet18_aug_geometric/resnet18_best.pt


## 4. Dataset partition integrity

The data preparation step produced **380 adaptation-training images** and **96 development images** from the original PlantDoc training partition. We verify the expected sizes, the fixed class IDs, and row-index non-overlap. These checks alone do not rule out visually duplicated images.

In [12]:
required_files = [
    "field_adapt_train.csv", "field_dev.csv", "field_test.csv", "lab_val.csv",
    "field_dev_protocol.json", "label_map.json",
]
for filename in required_files:
    if not (SPLITS_DIR / filename).is_file():
        raise FileNotFoundError(
            f"Missing {filename}. Commit/push the latest data/splits files from your Mac."
        )

manifests = {
    name: pd.read_csv(SPLITS_DIR / f"{name}.csv")
    for name in ("field_adapt_train", "field_dev", "field_test", "lab_val")
}
expected_sizes = {"field_adapt_train": 380, "field_dev": 96, "field_test": 47}
for name, expected in expected_sizes.items():
    frame = manifests[name]
    assert len(frame) == expected, (name, len(frame), expected)
    assert (frame["source"] == "plantdoc").all()
    assert frame["row_id"].is_unique
    assert set(frame["label_id"]) == set(range(5))
    assert (frame["class_name"].map(label_map) == frame["label_id"]).all()

train_ids = set(manifests["field_adapt_train"]["row_id"])
dev_ids = set(manifests["field_dev"]["row_id"])
test_ids = set(manifests["field_test"]["row_id"])
assert train_ids.isdisjoint(dev_ids)
assert train_ids.isdisjoint(test_ids)
assert dev_ids.isdisjoint(test_ids)

split_hashes = {
    name: hashlib.sha256((SPLITS_DIR / f"{name}.csv").read_bytes()).hexdigest()
    for name in ("field_adapt_train", "field_dev")
}

# Combine both partitions into a single DataFrame
split_df = pd.concat(
    [
        manifests["field_adapt_train"].assign(partition="adapt_train"),
        manifests["field_dev"].assign(partition="dev"),
    ],
    ignore_index=True
)

# Display class distribution across partitions
split_summary = pd.crosstab(
    split_df["class_name"],
    split_df["partition"]
)

display(split_summary)

print("Manifests verified; original Field Test will not be loaded for inference.")

partition,adapt_train,dev
class_name,,
Bacterial Spot,81,20
Early Blight,63,16
Healthy,44,11
Late Blight,80,21
Septoria Leaf Spot,112,28


Manifests verified; original Field Test will not be loaded for inference.


## 5. Image preprocessing and DataLoaders

Use **the same moderate adaptation-time augmentations** for A1 and A2: mild random cropping, horizontal flips, and color jitter. Use the original ImageNet-1k evaluation transform for Field Dev and Lab Validation. This isolates the effect of the *starting checkpoint* during adaptation.

In [13]:
WEIGHTS = ResNet18_Weights.IMAGENET1K_V1
val_transform = WEIGHTS.transforms()
mean, std = val_transform.mean, val_transform.std

train_transform = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BILINEAR),
    transforms.RandomResizedCrop(
        224, scale=(0.85, 1.0), ratio=(0.90, 1.10),
        interpolation=InterpolationMode.BILINEAR,
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.10)
    ], p=0.35),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

adapt_train_ds = PlantDiseaseDataset(
    SPLITS_DIR / "field_adapt_train.csv", transform=train_transform
)
field_dev_ds = PlantDiseaseDataset(
    SPLITS_DIR / "field_dev.csv", transform=val_transform
)
lab_val_ds = PlantDiseaseDataset(
    SPLITS_DIR / "lab_val.csv", transform=val_transform
)

train_loader = DataLoader(
    adapt_train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True
)
field_dev_loader = DataLoader(
    field_dev_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)
lab_val_loader = DataLoader(
    lab_val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

images, labels = next(iter(train_loader))
assert images.shape[1:] == (3, 224, 224)
print("Adapt train:", len(adapt_train_ds))
print("Field dev:", len(field_dev_ds))
print("Lab validation:", len(lab_val_ds))
print("Batch shape:", tuple(images.shape))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/542 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/476M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/516M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2578 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/402M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/453M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/54304 [00:00<?, ? examples/s]

Adapt train: 380
Field dev: 96
Lab validation: 1341
Batch shape: (32, 3, 224, 224)


## 6. Cross-partition duplicate audit (development data only)

**Purpose:** Flag identical or near-identical photographs across adaptation training and development partitions. Duplicate images can inflate the apparent benefit of adaptation. Exact duplicates block training; perceptual-hash candidates require manual review and are exported to CSV. No images from the original Field Test are inspected here.

In [14]:
import imagehash

source = adapt_train_ds.source_datasets["plantdoc"]

def image_fingerprints(frame):
    records = []
    for row in frame.itertuples(index=False):
        image = source[int(row.row_id)]["image"].convert("RGB")
        content_hash = hashlib.sha256(
            f"{image.size}".encode("utf-8") + image.tobytes()
        ).hexdigest()
        records.append({
            "row_id": int(row.row_id),
            "class_name": str(row.class_name),
            "exact_hash": content_hash,
            "phash": imagehash.phash(image),
        })
    return records

train_hashes = image_fingerprints(manifests["field_adapt_train"])
dev_hashes = image_fingerprints(manifests["field_dev"])
train_exact = {r["exact_hash"] for r in train_hashes}
exact_overlap = [r for r in dev_hashes if r["exact_hash"] in train_exact]
print("Exact cross-partition image duplicates:", len(exact_overlap))
if exact_overlap:
    raise ValueError(
        "Exact duplicate images found across Field Adapt Train and Field Dev. "
        "Fix the dataset split by image group and re-run all Field Dev comparisons."
    )

near_duplicates = [
    {
        "train_row_id": left["row_id"],
        "dev_row_id": right["row_id"],
        "train_class": left["class_name"],
        "dev_class": right["class_name"],
        "phash_distance": left["phash"] - right["phash"],
    }
    for left in train_hashes for right in dev_hashes
    if left["phash"] - right["phash"] <= 4
]
near_df = pd.DataFrame(near_duplicates)
near_path = OUTPUT_ROOT / "possible_near_duplicates.csv"
near_df.to_csv(near_path, index=False)
print("Potential near-duplicate pairs (pHash distance <= 4):", len(near_df))
print("Review candidates manually before treating Field Dev results as reliable.")
if len(near_df):
    display(near_df.sort_values("phash_distance").head(15))
print("Audit report:", near_path)

Exact cross-partition image duplicates: 2


ValueError: Exact duplicate images found across Field Adapt Train and Field Dev. Fix the dataset split by image group and re-run all Field Dev comparisons.

## 7. Model initialization and evaluation helpers

Each experiment starts from its own **frozen source checkpoint**, not from the other adapted experiment. For the small PlantDoc training set, BatchNorm running statistics remain frozen. A staged protocol first adapts the classifier head and then fine-tunes `layer4` plus the classifier.

In [ ]:
def load_source_model(path):
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    if checkpoint.get("architecture") != "resnet18":
        raise ValueError(f"Unexpected architecture: {path}")
    if checkpoint.get("class_names") != CLASS_NAMES:
        raise ValueError(f"Label mapping mismatch: {path}")
    if checkpoint.get("pretrained_weights") != "IMAGENET1K_V1":
        raise ValueError(f"Unexpected source preprocessing: {path}")
    model = resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    return model.to(DEVICE), checkpoint


def set_trainable(model, stage):
    for name, parameter in model.named_parameters():
        if stage == "head":
            parameter.requires_grad = name.startswith("fc.")
        elif stage == "last_block":
            parameter.requires_grad = name.startswith("layer4.") or name.startswith("fc.")
        else:
            raise ValueError(stage)


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    if training:
        # Frozen BN statistics reduce instability with a small field dataset.
        for module in model.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                module.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, truths, predictions = 0.0, [], []

    for images, targets in loader:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, targets)
            if training:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * len(targets)
        truths.extend(targets.detach().cpu().tolist())
        predictions.extend(logits.detach().argmax(dim=1).cpu().tolist())

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": float(accuracy_score(truths, predictions)),
        "macro_f1": float(f1_score(
            truths, predictions, labels=list(range(len(CLASS_NAMES))),
            average="macro", zero_division=0,
        )),
    }


def infer(model, loader):
    model.eval()
    truths, predictions, confidence = [], [], []
    with torch.inference_mode():
        for images, targets in loader:
            logits = model(images.to(DEVICE, non_blocking=True))
            prob = torch.softmax(logits, dim=1)
            score, pred = prob.max(dim=1)
            truths.extend(targets.tolist())
            predictions.extend(pred.cpu().tolist())
            confidence.extend(score.cpu().tolist())
    return np.asarray(truths), np.asarray(predictions), np.asarray(confidence)

## 8. Controlled adaptation experiments

The only difference between A1 and A2 is the **initial PlantVillage checkpoint**. Both use identical training data, augmentation, optimizer settings, seed, and validation protocol.

To avoid losing a strong source checkpoint, the unadapted model is retained as the initial candidate. Checkpoint selection uses **Field Dev macro-F1** only. Early stopping is applied during the last-block stage. Lab Validation is measured after selection to monitor potential forgetting, not to select the adapted checkpoint.

In [ ]:
def save_best(model, target_path, name, source_path, phase, epoch, field_f1):
    torch.save({
        "model_state_dict": model.state_dict(),
        "class_names": CLASS_NAMES,
        "architecture": "resnet18",
        "pretrained_weights": "IMAGENET1K_V1",
        "source_checkpoint": str(source_path),
        "adaptation_experiment": name,
        "best_field_dev_macro_f1": float(field_f1),
        "selected_phase": phase,
        "selected_epoch": int(epoch),
        "seed": SEED,
        "git_commit": GIT_SHA,
        "field_adapt_train_sha256": split_hashes["field_adapt_train"],
        "field_dev_sha256": split_hashes["field_dev"],
    }, target_path)


def adapt_experiment(name, source_path):
    # Restart random generators to make both schedules comparable.
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    experiment_dir = OUTPUT_ROOT / name
    experiment_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_file = experiment_dir / "resnet18_field_best.pt"
    model, original_checkpoint = load_source_model(source_path)
    before_field = run_epoch(model, field_dev_loader)
    before_lab = run_epoch(model, lab_val_loader)

    best_f1 = before_field["macro_f1"]
    selected_phase, selected_epoch = "source_unadapted", 0
    save_best(model, checkpoint_file, name, source_path, selected_phase, selected_epoch, best_f1)
    history = [{
        "phase": "source_unadapted", "epoch": 0,
        "train_loss": np.nan, "train_macro_f1": np.nan,
        "dev_loss": before_field["loss"],
        "dev_accuracy": before_field["accuracy"],
        "dev_macro_f1": before_field["macro_f1"],
    }]

    for stage, n_epochs in (("head", HEAD_EPOCHS), ("last_block", LAST_BLOCK_EPOCHS)):
        # Start the second phase from the best checkpoint of the first phase.
        if stage == "last_block":
            state = torch.load(checkpoint_file, map_location="cpu", weights_only=True)
            model.load_state_dict(state["model_state_dict"], strict=True)
        set_trainable(model, stage)
        if stage == "head":
            optimizer = torch.optim.AdamW(model.fc.parameters(), lr=1e-4, weight_decay=1e-4)
        else:
            optimizer = torch.optim.AdamW([
                {"params": model.layer4.parameters(), "lr": 1e-5},
                {"params": model.fc.parameters(), "lr": 1e-4},
            ], weight_decay=1e-4)

        no_improvement = 0
        for epoch in range(1, n_epochs + 1):
            train_metrics = run_epoch(model, train_loader, optimizer=optimizer)
            dev_metrics = run_epoch(model, field_dev_loader)
            history.append({
                "phase": stage, "epoch": epoch,
                "train_loss": train_metrics["loss"],
                "train_macro_f1": train_metrics["macro_f1"],
                "dev_loss": dev_metrics["loss"],
                "dev_accuracy": dev_metrics["accuracy"],
                "dev_macro_f1": dev_metrics["macro_f1"],
            })
            pd.DataFrame(history).to_csv(experiment_dir / "training_history.csv", index=False)
            if dev_metrics["macro_f1"] > best_f1 + 1e-6:
                best_f1 = dev_metrics["macro_f1"]
                selected_phase, selected_epoch = stage, epoch
                save_best(model, checkpoint_file, name, source_path, stage, epoch, best_f1)
                no_improvement = 0
            else:
                no_improvement += 1

            print(f"{name} | {stage} {epoch}/{n_epochs} | "
                  f"train F1={train_metrics['macro_f1']:.4f} | "
                  f"dev F1={dev_metrics['macro_f1']:.4f} | best={best_f1:.4f}")
            if stage == "last_block" and no_improvement >= EARLY_STOP_PATIENCE:
                print("Early stopping: no new best Field Dev macro-F1.")
                break

    best_checkpoint = torch.load(checkpoint_file, map_location="cpu", weights_only=True)
    model.load_state_dict(best_checkpoint["model_state_dict"], strict=True)
    after_field = run_epoch(model, field_dev_loader)
    after_lab = run_epoch(model, lab_val_loader)
    y_true, y_pred, scores = infer(model, field_dev_loader)
    assert np.isclose(after_field["macro_f1"], best_f1, atol=1e-9)

    report = classification_report(
        y_true, y_pred, labels=list(range(5)),
        target_names=CLASS_NAMES, digits=4, zero_division=0,
    )
    (experiment_dir / "field_dev_classification_report.txt").write_text(report, encoding="utf-8")
    predictions_df = manifests["field_dev"][
        ["source", "row_id", "class_name", "label_id"]
    ].copy()
    predictions_df["predicted_label_id"] = y_pred
    predictions_df["confidence"] = scores
    predictions_df["correct"] = y_pred == y_true
    predictions_df.to_csv(experiment_dir / "field_dev_predictions.csv", index=False)

    matrix = confusion_matrix(y_true, y_pred, labels=list(range(5)))
    fig, ax = plt.subplots(figsize=(8, 7))
    ConfusionMatrixDisplay(matrix, display_labels=CLASS_NAMES).plot(
        ax=ax, cmap="Blues", colorbar=False, xticks_rotation=30,
    )
    ax.set_title(f"ResNet18 {name} — Field Dev")
    fig.tight_layout()
    fig.savefig(experiment_dir / "field_dev_confusion_matrix.png", dpi=160)
    plt.close(fig)

    print(f"\n{name}: before Field Dev F1={before_field['macro_f1']:.4f}, "
          f"selected F1={after_field['macro_f1']:.4f}")
    print(f"Lab Val F1: {before_lab['macro_f1']:.4f} → {after_lab['macro_f1']:.4f}")
    print(f"Selected checkpoint: {selected_phase}, epoch {selected_epoch}")

    return {
        "experiment": name,
        "before_field_dev_macro_f1": before_field["macro_f1"],
        "after_field_dev_macro_f1": after_field["macro_f1"],
        "delta_field_dev_macro_f1": after_field["macro_f1"] - before_field["macro_f1"],
        "before_lab_val_macro_f1": before_lab["macro_f1"],
        "after_lab_val_macro_f1": after_lab["macro_f1"],
        "selected_phase": selected_phase,
        "selected_epoch": selected_epoch,
        "checkpoint": str(checkpoint_file),
    }

## 9. Run A1 and A2

Run both experiments without modifying the development partition, training schedule, or hyperparameters between them. Results are exploratory and are subject to selection optimism because Field Dev is used to select checkpoints.

In [ ]:
experiment_results = []
for name, source_path in CHECKPOINTS.items():
    experiment_results.append(adapt_experiment(name, source_path))

summary_df = pd.DataFrame(experiment_results)
summary_df.to_csv(OUTPUT_ROOT / "adaptation_summary.csv", index=False)
display(summary_df.drop(columns="checkpoint"))
print("Summary:", OUTPUT_ROOT / "adaptation_summary.csv")

## 10. Visualize improvement and forgetting

Compare **before vs. selected checkpoint** on Field Dev and Lab Validation. An adapted model can improve field performance while losing accuracy in the source domain (catastrophic forgetting).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
labels = ["A1 Baseline", "A2 Geometric"]
x = np.arange(len(labels))
width = 0.34

axes[0].bar(x - width/2, summary_df["before_field_dev_macro_f1"], width, label="Before")
axes[0].bar(x + width/2, summary_df["after_field_dev_macro_f1"], width, label="Selected")
axes[0].set_title("Field Dev Macro-F1")
axes[0].set_ylim(0, 1)
axes[0].set_xticks(x, labels)
axes[0].legend()

axes[1].bar(x - width/2, summary_df["before_lab_val_macro_f1"], width, label="Before")
axes[1].bar(x + width/2, summary_df["after_lab_val_macro_f1"], width, label="Selected")
axes[1].set_title("Lab Validation Macro-F1")
axes[1].set_ylim(0, 1)
axes[1].set_xticks(x, labels)
axes[1].legend()

fig.suptitle("Supervised Field Adaptation — ResNet18")
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "adaptation_comparison.png", dpi=160)
plt.show()

## 11. Study notes and interpretation

After running the notebook, document:

1. Whether each adapted model improved **Field Dev macro-F1** over its original source checkpoint.
2. Whether the gain came from improved recall across multiple classes or only a few images.
3. Whether **Lab Validation macro-F1** decreased after adaptation.
4. Whether exact or perceptual duplicate-image warnings require changes to the partitions.
5. The limitations of 380 adaptation-training images, 96 development images, and checkpoint selection on the development set.

**Do not report the original Field Test as a fresh independent confirmation.** An additional external test set or prospective field collection is required for confirmatory generalization claims. The same controlled adaptation protocol can later be repeated with ViT-Small.